# Meteosat-9 reflectance animation around Maroantsetra flood dates

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johrosa/srwi/blob/main/wekeo_meteosat_animation_maroantsetra_colab.ipynb)

This notebook builds Meteosat-9 IODC cloud-motion animations around Maroantsetra using **visible reflectance / natural-colour EUMETView WMS layers**, not rainfall products.

Default layer: `msg_iodc:vis006`, the Meteosat SEVIRI visible 0.6 micrometer channel. You can switch to `msg_iodc:rgb_natural` or `msg_iodc:rgb_eview` for rendered RGB imagery.

## 1. Install and imports

In [ ]:
!pip -q install requests pillow imageio hda rasterio


In [ ]:
import getpass
import json
import math
import os
import re
import time
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
from pathlib import Path

import imageio.v2 as imageio
import numpy as np
import pandas as pd
import requests
import rasterio
from IPython.display import Image, display
from PIL import Image as PILImage, ImageDraw, ImageFont
from rasterio.transform import from_bounds


## 2. Parameters

In [ ]:
MAROANTSETRA_LON = 49.7333
MAROANTSETRA_LAT = -15.4333
BBOX = [47.4, -17.6, 52.1, -13.2]  # west, south, east, north

EUMETVIEW_WMS_URL = "https://view.eumetsat.int/geoserver/wms"
REFLECTANCE_LAYER = "msg_iodc:vis006"
# Other useful non-rainfall layers:
# - msg_iodc:rgb_natural
# - msg_iodc:rgb_eview
# - msg_iodc:rgb_microphysics
# - msg_iodc:ir108 for day/night thermal cloud tracking, not reflectance

FLOOD_EVENTS = [
    {
        "name": "Cyclone Herold - Maroantsetra floods",
        "start": "2020-03-13T00:00:00Z",
        "end": "2020-03-18T23:59:59Z",
    },
    {
        "name": "Cyclone Gamane - north-east Madagascar floods",
        "start": "2024-03-26T00:00:00Z",
        "end": "2024-03-29T23:59:59Z",
    },
]

STEP_MINUTES = 15
WIDTH = 720
HEIGHT = 720
GIF_FPS = 6
MAX_FRAMES_PER_GIF = 96
OUTPUT_DIR = Path("meteosat_reflectance_wms")
WRITE_GEOTIFF_FRAMES = True
OUTPUT_CRS = "EPSG:4326"
PAUSE_SECONDS = 0.15
HTTP_MAX_RETRIES = 4
HTTP_BACKOFF_SECONDS = 2

# Daytime visible reflectance is dark at night. Restrict to daylight hours over Madagascar if desired.
DAYLIGHT_ONLY = True
DAYLIGHT_UTC_START_HOUR = 3
DAYLIGHT_UTC_END_HOUR = 15


## 3. WMS layer discovery

In [ ]:
def parse_datetime(value):
    value = value.replace("Z", "+00:00")
    dt = datetime.fromisoformat(value)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def local_name(tag):
    return tag.rsplit("}", 1)[-1]


def first_child_text(element, child_name):
    for child in element:
        if local_name(child.tag) == child_name and child.text:
            return child.text.strip()
    return None


def layer_time_dimension(layer):
    for child in layer:
        tag = local_name(child.tag)
        if tag not in {"Dimension", "Extent"}:
            continue
        if child.attrib.get("name") == "time" and child.text:
            return child.text.strip()
    return None


def parse_layer_time_extent(value):
    if not value:
        return None, None
    parts = [part.strip() for part in value.split("/")]
    if len(parts) < 2:
        return None, None
    return parse_datetime(parts[0]), parse_datetime(parts[1])


def list_wms_layers(pattern="msg_iodc:(vis|rgb|hrv|ir108)|reflectance|natural"):
    response = requests.get(
        EUMETVIEW_WMS_URL,
        params={"SERVICE": "WMS", "VERSION": "1.3.0", "REQUEST": "GetCapabilities"},
        timeout=60,
    )
    response.raise_for_status()
    root = ET.fromstring(response.content)
    regex = re.compile(pattern, re.IGNORECASE) if pattern else None
    rows = []

    for layer in root.iter():
        if local_name(layer.tag) != "Layer":
            continue
        name = first_child_text(layer, "Name")
        if not name:
            continue
        title = first_child_text(layer, "Title") or ""
        text = f"{name} {title}"
        if regex is None or regex.search(text):
            rows.append({"name": name, "title": title, "time_extent": layer_time_dimension(layer)})

    return pd.DataFrame(rows)


layers_df = list_wms_layers()
display(layers_df)

selected_layer_rows = layers_df[layers_df["name"] == REFLECTANCE_LAYER]
if selected_layer_rows.empty:
    print("Selected layer not shown by the discovery filter. It may still exist, or adjust REFLECTANCE_LAYER.")
    LAYER_START = None
    LAYER_END = None
else:
    LAYER_START, LAYER_END = parse_layer_time_extent(selected_layer_rows.iloc[0].get("time_extent"))
    if LAYER_START and LAYER_END:
        print(f"{REFLECTANCE_LAYER} available from {LAYER_START:%Y-%m-%d %H:%M UTC} to {LAYER_END:%Y-%m-%d %H:%M UTC}")
    else:
        print(f"No time extent advertised for {REFLECTANCE_LAYER}; dates will not be clipped automatically.")


## 4. Download reflectance frames

In [ ]:
def safe_name(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")


def iter_times(start, end, step_minutes):
    current = start
    step = timedelta(minutes=step_minutes)
    while current <= end:
        if not DAYLIGHT_ONLY or DAYLIGHT_UTC_START_HOUR <= current.hour < DAYLIGHT_UTC_END_HOUR:
            yield current
        current += step


def split_times(times, max_frames):
    return [times[i:i + max_frames] for i in range(0, len(times), max_frames)]


def clip_event_to_layer_extent(event):
    start = parse_datetime(event["start"])
    end = parse_datetime(event["end"])
    original_start, original_end = start, end

    if LAYER_START:
        start = max(start, LAYER_START)
    if LAYER_END:
        end = min(end, LAYER_END)

    if start > end:
        return None, {
            "event": event["name"],
            "status": "skipped",
            "reason": f"outside {REFLECTANCE_LAYER} WMS availability",
            "requested_start": original_start,
            "requested_end": original_end,
            "used_start": None,
            "used_end": None,
            "frames": 0,
        }

    status = "clipped" if (start != original_start or end != original_end) else "ok"
    return {**event, "start": start, "end": end}, {
        "event": event["name"],
        "status": status,
        "reason": "",
        "requested_start": original_start,
        "requested_end": original_end,
        "used_start": start,
        "used_end": end,
        "frames": None,
    }


def wms_params(layer, timestamp):
    return {
        "SERVICE": "WMS",
        "VERSION": "1.1.1",
        "REQUEST": "GetMap",
        "LAYERS": layer,
        "STYLES": "",
        "SRS": "EPSG:4326",
        "BBOX": ",".join(str(v) for v in BBOX),
        "WIDTH": str(WIDTH),
        "HEIGHT": str(HEIGHT),
        "FORMAT": "image/png",
        "TRANSPARENT": "false",
        "TIME": timestamp.strftime("%Y-%m-%dT%H:%M:%SZ"),
    }


def annotate_frame(path, label):
    image = PILImage.open(path).convert("RGBA")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    margin = 10
    padding = 6
    text_box = draw.textbbox((0, 0), label, font=font)
    box = (
        margin,
        margin,
        margin + text_box[2] - text_box[0] + 2 * padding,
        margin + text_box[3] - text_box[1] + 2 * padding,
    )
    draw.rectangle(box, fill=(0, 0, 0, 170))
    draw.text((margin + padding, margin + padding), label, fill=(255, 255, 255, 255), font=font)
    image.convert("RGB").save(path)




def png_to_geotiff(png_path, tif_path, bbox, crs=OUTPUT_CRS):
    west, south, east, north = bbox
    image = PILImage.open(png_path).convert("RGB")
    array = np.array(image)
    height, width = array.shape[:2]
    transform = from_bounds(west, south, east, north, width, height)

    tif_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        tif_path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=3,
        dtype=array.dtype,
        crs=crs,
        transform=transform,
    ) as dst:
        dst.write(array[:, :, 0], 1)
        dst.write(array[:, :, 1], 2)
        dst.write(array[:, :, 2], 3)
    return tif_path


def get_wms_response(layer, timestamp):
    params = wms_params(layer, timestamp)
    last_exc = None
    for attempt in range(1, HTTP_MAX_RETRIES + 1):
        try:
            response = requests.get(EUMETVIEW_WMS_URL, params=params, timeout=90)
            if response.status_code in {429, 500, 502, 503, 504} and attempt < HTTP_MAX_RETRIES:
                wait = HTTP_BACKOFF_SECONDS * attempt
                print(f"retry {attempt}/{HTTP_MAX_RETRIES} for {timestamp:%Y-%m-%d %H:%M UTC}: HTTP {response.status_code}; wait {wait}s")
                time.sleep(wait)
                continue
            response.raise_for_status()
            return response
        except requests.RequestException as exc:
            last_exc = exc
            if attempt >= HTTP_MAX_RETRIES:
                raise
            wait = HTTP_BACKOFF_SECONDS * attempt
            print(f"retry {attempt}/{HTTP_MAX_RETRIES} for {timestamp:%Y-%m-%d %H:%M UTC}: {exc}; wait {wait}s")
            time.sleep(wait)
    raise last_exc


def download_frame(layer, timestamp, output_path):
    response = get_wms_response(layer, timestamp)
    content_type = response.headers.get("content-type", "")
    if "xml" in content_type.lower() or response.content[:100].lstrip().startswith(b"<"):
        raise RuntimeError(response.text[:1000])

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_bytes(response.content)
    annotate_frame(output_path, f"{layer} {timestamp:%Y-%m-%d %H:%M UTC}")
    return output_path


## 5. Build GIF animations

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
gif_paths = []
frame_rows = []
event_status_rows = []

for original_event in FLOOD_EVENTS:
    event, status_row = clip_event_to_layer_extent(original_event)
    if event is None:
        event_status_rows.append(status_row)
        if LAYER_START and LAYER_END:
            print(
                f"Skipping {original_event['name']}: requested dates are outside "
                f"{REFLECTANCE_LAYER} WMS availability "
                f"({LAYER_START:%Y-%m-%d %H:%M UTC} to {LAYER_END:%Y-%m-%d %H:%M UTC})."
            )
        else:
            print(f"Skipping {original_event['name']}: dates are outside the selected layer availability.")
        continue

    event_name = safe_name(event["name"])
    start = event["start"]
    end = event["end"]
    times = list(iter_times(start, end, STEP_MINUTES))
    chunks = split_times(times, MAX_FRAMES_PER_GIF)
    status_row["frames"] = len(times)
    event_status_rows.append(status_row)
    print(event["name"], "status:", status_row["status"], "frames:", len(times), "chunks:", len(chunks))

    for chunk_idx, chunk_times in enumerate(chunks, start=1):
        frame_paths = []
        frame_dir = OUTPUT_DIR / event_name / f"part_{chunk_idx:02d}"

        for timestamp in chunk_times:
            frame_path = frame_dir / f"{timestamp:%Y%m%dT%H%M%SZ}.png"
            try:
                download_frame(REFLECTANCE_LAYER, timestamp, frame_path)
                geotiff_path = None
                if WRITE_GEOTIFF_FRAMES:
                    geotiff_path = frame_path.with_suffix(".tif")
                    png_to_geotiff(frame_path, geotiff_path, BBOX)
                frame_paths.append(frame_path)
                frame_rows.append({
                    "event": event["name"],
                    "time": timestamp,
                    "png": str(frame_path),
                    "geotiff": str(geotiff_path) if geotiff_path else "",
                    "layer": REFLECTANCE_LAYER,
                    "crs": OUTPUT_CRS,
                    "bbox_west": BBOX[0],
                    "bbox_south": BBOX[1],
                    "bbox_east": BBOX[2],
                    "bbox_north": BBOX[3],
                })
                print("downloaded", frame_path, "geotiff:", geotiff_path)
            except Exception as exc:
                print("failed", timestamp, exc)
            if PAUSE_SECONDS:
                time.sleep(PAUSE_SECONDS)

        if frame_paths:
            images = [imageio.imread(path) for path in frame_paths]
            suffix = f"_part{chunk_idx:02d}" if len(chunks) > 1 else ""
            gif_path = OUTPUT_DIR / f"{event_name}{suffix}_{REFLECTANCE_LAYER.replace(':', '_')}.gif"
            imageio.mimsave(gif_path, images, duration=1 / GIF_FPS)
            gif_paths.append(gif_path)
            print("GIF:", gif_path)
            display(Image(filename=str(gif_path)))

frames_df = pd.DataFrame(frame_rows)
event_status_df = pd.DataFrame(event_status_rows)
metadata_csv = OUTPUT_DIR / "frames_metadata.csv"
frames_df.to_csv(metadata_csv, index=False)
print("Frame metadata:", metadata_csv)
display(event_status_df)
display(frames_df.head())
display(pd.DataFrame({"gif": [str(p) for p in gif_paths]}))


## 6. Optional: WEkEO HDA metadata for raw SEVIRI image data

WEkEO HDA currently exposes many MSG/IODC derived products, but the raw High Rate SEVIRI IODC product may return `404` depending on the catalogue exposure/account. This optional cell checks the raw reflectance/radiance product id only. It does not use rainfall products.

In [ ]:
RUN_WEKEO_HDA_CHECK = False
RAW_SEVIRI_DATASET_ID = "EO:EUM:DAT:MSG:HRSEVIRI-IODC"

if RUN_WEKEO_HDA_CHECK:
    from hda import Client, Configuration

    username = os.environ.get("WEKEO_USERNAME") or input("WEkEO username: ")
    password = os.environ.get("WEKEO_PASSWORD") or getpass.getpass("WEkEO password: ")
    hda_client = Client(config=Configuration(user=username, password=password))

    try:
        dataset_info = hda_client.dataset(RAW_SEVIRI_DATASET_ID)
        print("Raw SEVIRI dataset available in WEkEO HDA:", RAW_SEVIRI_DATASET_ID)
        display(dataset_info)
    except Exception as exc:
        print("Raw SEVIRI dataset is not exposed through this WEkEO HDA endpoint/account:", RAW_SEVIRI_DATASET_ID)
        print(type(exc).__name__, exc)

## Notes

- This notebook uses reflectance/visible WMS imagery by default (`msg_iodc:vis006`), not precipitation.
- `msg_iodc:rgb_natural` and `msg_iodc:rgb_eview` are useful rendered visible/RGB alternatives.
- Visible reflectance is daytime-only; keep `DAYLIGHT_ONLY = True` for cleaner animations.
- For night-time cloud motion use `msg_iodc:ir108`, but that is thermal infrared brightness, not reflectance.
- The WEkEO HDA optional cell checks raw SEVIRI availability only and deliberately avoids rainfall datasets.